# `hscmap` Python Package Tutorial

This notebook is a tutorial for the `hscmap` Python package. [`hscMap`](https://hscmap.mtk.nao.ac.jp/hscMap5/) is a web application for astronomical image viewing that handles wide-area data from HSC-SSP. The `hscmap` Python package is a library for using `hscMap` from Python.

<!--
The `hscmap` Python package provides the following features:
This function allows users to manipulate the viewer’s display area, providing capabilities to both retrieve and set the visible region.
Manipulation of Displayed Data Sets:
The package offers the flexibility to add or remove data sets displayed on the viewer, including the ability to display or hide HiPS (Hierarchical Progressive Surveys) data.
Catlog Overlay:
This feature enables the overlaying of a catalog—a list of coordinates—onto the viewer. Users can interact with the markers corresponding to these coordinates, with the ability to select them directly through the viewer. Moreover, the selection state of these markers can be managed and retrieved from Python.
Region Annotation:
Users can annotate the viewer with various shapes such as circles, rectangles, texts, and custom arbitrary shapes, facilitating detailed visualization and analysis of specific areas.
Monitoring the Viewer State:
The “Monitoring the Viewer State” feature lets Python detect and respond to changes in the hscMap viewer caused by user interactions. By registering callback functions. -->

First, run the following cell. This command installs the libraries required to run this notebook.

In [ ]:
%pip install hscmap
%pip install requests

## Creating a Viewer Window

Run the following code to create a viewer window.

In [1]:
from hscmap import Window

w = Window()

The `Window` class is the central class of this package, creating a viewer window for `hscmap`.

See more details for the `Window` class in the [API documentation](/hscMap5/python/docs/autodoc/hscmap.window.html).

## Camera Control

You can set the display area of the viewer using `w.jump_to` and `w.camera`.

In [ ]:
# Go to Abell1689
w.jump_to(197.87, -1.342, fov=0.2)

In [ ]:
# Or, if you want to use the camera directly:
w.camera.jump_to(197.87, -1.342, fov=0.4)

You cat get and set the camera position using `w.camera`.

In [ ]:
w.camera.center

In [ ]:
w.camera.fov

In [ ]:
w.camera.fov *= 2

See more details for `Window.camera` in the [API documentation](/hscMap5/python/docs/autodoc/hscmap.window.html#hscmap.window.Window.camera).

## Dataset Control

You can control the data displayed through `w.dataset`.

### Tile Layers

Currently, "PDR3 Wide" and "PDR3 DUD" are available as tile layers.

In [ ]:
# Jump to area where both PDR3 Wide and PDR3 DUD are available.
w.jump_to(353, 1, fov=2.5)

You can show and hide the dataset by:

In [ ]:
pdr3_dud = w.dataset.tile_layers['PDR3 DUD']
pdr3_dud.visible = False

In [ ]:
# Show PDR3 Wide again
pdr3_dud.visible = True

### HiPS Layers

You can search and display HiPS data registered at [The Strasbourg astronomical Data Center's HiPS Registry](http://aladin.cds.unistra.fr/hips/list).

In [ ]:
# Zoom out to see the whole sky
w.jump_to(280, 5, fov=230)

In [ ]:
# Search for HiPS data containing 'panstarrs' in the name
panstarrs_query_results = w.dataset.hips.find_by_name('panstarrs')
panstarrs_query_results

In [ ]:
# Show the first result
w.dataset.hips.base_url = panstarrs_query_results[0].hips_service_url

In [ ]:
# Show HiPS information
w.dataset.hips.properties

In [ ]:
w.dataset.hips.clear()

See more details for the `Window.dataset` attribute in the [API Reference](/hscMap5/python/docs/autodoc/hscmap.window.html#hscmap.window.Window.dataset).

## Catalog Overlay

You can overlay catalogs on hscMap.
(Here "catalog" means a list of coordinates.)

In [ ]:
# Generate a random catalog
import numpy

def make_random_catalog(
    n: int,
    min_ra: float = -1,
    max_ra: float = 1,
    min_dec: float = -1,
    max_dec: float = 1,
):
    ra = numpy.random.uniform(min_ra, max_ra, n)
    ra.sort()
    dec = numpy.random.uniform(min_dec, max_dec, n)
    return ra, dec

ra, dec = make_random_catalog(1000)

In [ ]:
w.jump_to(0, 0, fov=4)

In [ ]:
catalog = w.catalogs.new(ra, dec)

In [ ]:
# You can change the color of the markers
catalog.color = [0, 1, 0, 0.5]

In [ ]:
# You can change the marker type
catalog.marker = 'triangle'

In [ ]:
# To know the available marker types, set an arbitrary value
catalog.marker = '?'

Click on some of the overlaid markers on the viewer to select them.
You can get the indices of the selected markers by:

In [ ]:
catalog.selected_indices

In [ ]:
# Select the first 20 points
catalog.selected_indices = range(100)

In [ ]:
# Clear all the catalogs
w.catalogs.clear()

See more details for the `Window.catalogs` attribute in the [API Reference](/hscMap5/python/docs/autodoc/hscmap.window.html#hscmap.window.Window.catalogs).

# Region Annotation

You can annotate the viewer with various shapes such as circles, rectangles, texts, and custom arbitrary shapes, facilitating detailed visualization and analysis of specific areas.

In [ ]:
w.jump_to(0, 0, fov=0.5)

### Text

In [ ]:
t = w.regions.new_text(position=(0, 0), text='Hello')

In [ ]:
t.name = 'Hello World!'

In [ ]:
import time

for i in range(10):
    t.position = (0, 0.01 * i)
    time.sleep(0.1)

### Circle

In [ ]:
c = w.regions.new_circle(center=(0, 0), radius=0.1)

In [ ]:
c.radius = 0.15

In [ ]:
c.color = [0, 1, 1, 0.5]

### Line

In [ ]:
l = w.regions.new_line(start=(0, 0), end=(0.1, 0.2))

In [ ]:
l.show_label = False

In [ ]:
l.show_label = True

### Arbitrary Shape

In [ ]:
# I know this is not a practical example, but it's just for demonstration.
from hscmap import Vec3
from hscmap.shape import Polyline
import math


def Lissajous_curve(a: int, b: int, size=1.0, n: int = 1000):
    ts = [2 * math.pi * i / n for i in range(n)]
    for t in ts:
        x = 1
        y = size * math.sin(a * t)
        z = size * math.cos(b * t)
        yield Vec3(x, y, z)


lissajous = Polyline(
    close=True,
    color=[0, 1, 1, 0.75],
    points=[*Lissajous_curve(2, 3, size=0.01)],
)


w.camera.jump_to(0, 0, fov=2)
w.regions.from_shape(shape=lissajous)


In [ ]:
w.regions.clear()

See more details for the `Window.regions` attribute in the [API Reference](/hscMap5/python/docs/autodoc/hscmap.window.html#hscmap.window.Window.regions).

## Save image

You can get the current viewer image by:

In [ ]:
image = w.snapshot_image(aspect_ratio=1.5)
image

### Save the image to disk

In [ ]:
from pathlib import Path

Path('image.png').write_bytes(image.data)

## Lock windows

You can synchronize the display positions of two windows.

In [ ]:
# Create the second window
w2 = Window(layout='split-bottom', title='hscMap2')

In [ ]:
# Lock position and fov of the first window to the second window
unlock = w.lock(w2)

You can also display data from another survey side by side.

In [ ]:
w.jump_to(7.541, 2.092, fov=0.2)
# Hide PDR3
w2.dataset.tile_layers['PDR3 Wide'].visible = False
# Search for AKARI data
akari = w2.dataset.hips.find_by_name('akari*fis')
w2.dataset.hips.base_url = akari[0].hips_service_url

In [ ]:
unlock()  # Unlock the positions and fovs
w2.close()  # Close the second window

## Monitoring the Viewer State

The "Monitoring the Viewer State" feature lets Python detect and respond to changes in the `hscMap` viewer caused by user interactions. By registering callback functions.

In [ ]:
from hscmap import SkyCoord, Angle
from hscmap.shape import Grid

In [ ]:
circle = w.regions.new_circle(
    name='DraggableGrid/Handle',
    center=w.camera.center,
    radius=0.75,
    color=[1, 1, 0, 1],
)

w.camera.fov = 2
def draw_grid(center):
    shape = Grid(
        center=SkyCoord.from_degree(*center),
        color=[0, 0.75, 1, 1],
        width=Angle.from_degree(1.36),
        height=Angle.from_degree(1.38),
        div_x=6,
        div_y=4,
    )
    grid = w.regions.from_shape(name='DraggableGrid/Grid', shape=shape)

    def cleanup():
        grid.delete()

    return cleanup


def watch_on():
    return circle.center


cleanup = None


def on_change():
    global cleanup
    if cleanup is not None:
        cleanup()
    cleanup = draw_grid(circle.center)
    circle.surface()  # Bring the circle to the front


on_change()
watcher = w.watchers.new(watch_on=watch_on, on_change=on_change)

In [ ]:
watcher.unwatch()  # Stop watching

See more details for `Window.watchers` in the [API Reference](/hscMap5/python/docs/autodoc/hscmap.window.html#hscmap.window.Window.watchers).